# Image Generation Using Denoising Diffusion Implicit Models (DDIM)

## Reqs

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Lambda, RandomHorizontalFlip

from torchsummary import summary
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
from tqdm import tqdm
import numpy as np
from PIL import Image
import os
import kagglehub
import matplotlib.animation as animation
from matplotlib import rc
import seaborn as sns
from IPython.display import clear_output
from copy import deepcopy
from torch.amp import autocast, GradScaler
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
!pip install torchinfo
clear_output()

## Data

In [ ]:
# Download the dataset from Kaggle
path = kagglehub.dataset_download("soumikrakshit/anime-faces")
print("Path to dataset files:", path)

# The dataset path will be the downloaded path
dataset_path = os.path.join(path,'data')

In [ ]:
#@title dataset
IMAGE_SIZE = 64  # Anime faces will be resized to 64x64
transform_steps = Compose([
    Resize(IMAGE_SIZE),               # Resize to a consistent 64x64
    CenterCrop(IMAGE_SIZE),           # Crop to the center to ensure dimensions
    RandomHorizontalFlip(p=0.5),      # Flip left-right 50% of the time — free data augmentation
    ToTensor(),                       # Convert image to a PyTorch tensor (scales to [0, 1])
    Lambda(lambda t: (t * 2) - 1)     # Normalize to the range [-1, 1]
])

class AnimeFacesDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(root_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, 0  # Return 0 as dummy label since we don't need labels

BATCH_SIZE = 128
VAL_FRACTION = 0.05   # small val set — 5%

anime_dataset = AnimeFacesDataset(root_dir=dataset_path, transform=transform_steps)

# split
val_size = int(len(anime_dataset) * VAL_FRACTION)
train_size = len(anime_dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(
    anime_dataset, [train_size, val_size]
)

workers=os.cpu_count()
train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=workers, pin_memory=True, drop_last=True,   # drop_last avoids batch-size-1 BatchNorm crash
)
val_dataloader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=workers, pin_memory=True, drop_last=False,
)

print(f"Train: {train_size} images | Val: {val_size} images | Workers: {workers}")

In [ ]:
#@title displaying images
figure = plt.figure(figsize=(14, 14))
cols, rows = 7, 1
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(anime_dataset), size=(1,)).item()
    img, _ = anime_dataset[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(f"Anime Face {i}")
    plt.axis("off")
    # Convert from [-1,1] back to [0,1] for display and clamp to valid range
    img_display = np.clip((img.numpy() + 1) / 2, 0, 1)
    plt.imshow(img_display.transpose(1, 2, 0))
plt.show()

## Architecture

In [ ]:
# !pip install diffusers accelerate
from diffusers import UNet2DModel, DDPMScheduler

IMAGE_SIZE = 64

model = UNet2DModel(
    sample_size=IMAGE_SIZE,
    in_channels=3, out_channels=3,
    layers_per_block=3,
    block_out_channels=(64, 128, 256, 256),        # base 64, mults (1,2,4) -> 3 blocks: 64->32->16
    down_block_types=(
        "DownBlock2D",       # 64x64
        "DownBlock2D",       # 32x32
        "AttnDownBlock2D",   # 16x16
        "AttnDownBlock2D",   # 8   — attention (new deepest level)
    ),
    up_block_types=(
        "AttnUpBlock2D",     # 8x8
        "AttnUpBlock2D",     # 16x16
        "UpBlock2D",         # 32x32
        "UpBlock2D",         # 64x64
    ),
    norm_num_groups=32,
).to(device)

print(f"param count {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from diffusers import DDIMScheduler

# Training scheduler: full 1000-step noise schedule, used for add_noise + sampling t
noise_scheduler = DDIMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",
)

# Sampling scheduler: same config, but reduced to 50 reverse steps for fast generation
sampling_scheduler = DDIMScheduler.from_config(noise_scheduler.config)
sampling_scheduler.set_timesteps(50)

@torch.no_grad()
def sample_ddim(model, scheduler, n=16, image_size=64):
    model.eval()
    img = torch.randn(n, 3, image_size, image_size, device=device)
    for t in scheduler.timesteps:                 # 50 iterations
        pred = model(img, t).sample
        img = scheduler.step(pred, t, img).prev_sample
    return img

imgs = sample_ddim(model, sampling_scheduler, n=16)
imgs = (imgs.clamp(-1, 1) + 1) / 2

In [ ]:
#@title hyperparams
LEARNING_RATE = 2e-4
EPOCHS = 50
GRAD_CLIP = 1.0

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-6)
total_steps = EPOCHS * len(train_dataloader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LEARNING_RATE, total_steps=total_steps, pct_start=0.05
)

torch.backends.cudnn.benchmark = True
scaler = GradScaler()
train_losses, epoch_losses, val_losses = [], [], []

def diffusion_loss(model, clean, gamma=5.0):
    """Noise-prediction loss with Min-SNR-gamma weighting."""
    clean = clean.to(device, non_blocking=True)
    noise = torch.randn_like(clean)
    t = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                      (clean.shape[0],), device=device).long()
    noisy = noise_scheduler.add_noise(clean, noise, t)
    pred = model(noisy, t).sample

    # per-sample MSE (mean over C,H,W, keep the batch dim)
    mse = F.mse_loss(pred, noise, reduction="none").mean(dim=[1, 2, 3])

    # SNR(t) = alphas_cumprod / (1 - alphas_cumprod)
    acp = noise_scheduler.alphas_cumprod.to(device)[t]
    snr = acp / (1 - acp)

    # Min-SNR-gamma weight: min(snr, gamma) / snr
    weight = torch.clamp(snr, max=gamma) / snr

    return (weight * mse).mean()

In [ ]:
EPOCHS = 30

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-6)

total_steps = EPOCHS * len(train_dataloader) + 10   # small buffer absorbs off-by-one
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LEARNING_RATE, total_steps=total_steps, pct_start=0.05
)

In [ ]:
from IPython.display import clear_output
import matplotlib.pyplot as plt

SAMPLE_EVERY = 2        # generate a preview grid every N epochs
FLUSH_EVERY = 20         # clear previous cell output every N epochs (keeps notebook light)

def preview_samples(model, scheduler, n=8):
    """Generate and show a small DDIM grid, then restore train mode."""
    model.eval()
    with torch.no_grad():
        img = torch.randn(n, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
        for t in scheduler.timesteps:
            pred = model(img, t).sample
            img = scheduler.step(pred, t, img).prev_sample
    img = (img.clamp(-1, 1) + 1) / 2
    img = img.cpu()

    cols = n
    fig, axes = plt.subplots(1, cols, figsize=(2 * cols, 2.2))
    for ax, im in zip(axes, img):
        ax.imshow(im.permute(1, 2, 0).numpy()); ax.axis("off")
    plt.tight_layout(); plt.show()
    model.train()          # IMPORTANT: back to train mode after sampling


model = model.to(device)
for i in range(1, EPOCHS+1):
    # ===== FLUSH ===== (clear accumulated output periodically)
    if i % FLUSH_EVERY == 0:
        clear_output(wait=True)

    # ===== TRAIN =====
    model.train()
    epoch_loss, batch_count = 0, 0
    pbar = tqdm(train_dataloader)
    for inputs, _ in pbar:
        optimizer.zero_grad(set_to_none=True)

        with autocast(device):
            loss = diffusion_loss(model, inputs)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        batch_loss = loss.item()
        train_losses.append(batch_loss)
        epoch_loss += batch_loss
        batch_count += 1
        pbar.set_description(f"Epoch {i}/{EPOCHS}")
        pbar.set_postfix({"loss": batch_loss,
                          "avg": epoch_loss/batch_count,
                          "lr": scheduler.get_last_lr()[0]})

    avg = epoch_loss / batch_count
    epoch_losses.append(avg)

    # ===== VALIDATE =====
    model.eval()
    val_loss, val_batches = 0, 0
    with torch.no_grad():
        for inputs, _ in val_dataloader:
            with autocast(device):
                loss = diffusion_loss(model, inputs)
            val_loss += loss.item()
            val_batches += 1

    avg_val = val_loss / val_batches
    val_losses.append(avg_val)
    print(f"Epoch {i} - Train Loss: {avg:.6f} | Val Loss: {avg_val:.6f}")

    # ===== PREVIEW ===== (watch quality climb past the loss plateau)
    if i % SAMPLE_EVERY == 0:
        print(f"--- DDIM samples at epoch {i} ---")
        preview_samples(model, sampling_scheduler, n=8)

In [ ]:
#@title plotting training
# Set a professional plotting style
sns.set_theme(style="whitegrid")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Plot batch-wise losses (Training only)
ax1.plot(train_losses, color='#4C72B0', alpha=0.6, label='Batch Loss')
ax1.set_title('Training Loss (Per Batch)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Batch Step')
ax1.set_ylabel('Loss')
ax1.legend()

# Plot epoch-wise average losses (Training vs Validation)
ax2.plot(range(1, len(epoch_losses) + 1), epoch_losses, 'o-', color='#4C72B0',
         linewidth=2, markersize=6, label='Train Loss')
ax2.plot(range(1, len(val_losses) + 1), val_losses, 's--', color='#C44E52',
         linewidth=2, markersize=6, label='Val Loss')

ax2.set_title('Training & Validation Loss (Per Epoch)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Average Loss')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
#@title Sampling function (DDIM, uses the diffusers scheduler)
@torch.no_grad()
def sample_ddim(model, scheduler, n=16, image_size=64, keep_history=False):
    model.eval()
    img = torch.randn(n, 3, image_size, image_size, device=device)
    history = [img.cpu()] if keep_history else None
    for t in scheduler.timesteps:                 # 50 steps (set by set_timesteps)
        pred = model(img, t).sample               # .sample -> tensor
        img = scheduler.step(pred, t, img).prev_sample
        if keep_history:
            history.append(img.cpu())
    return (img, history) if keep_history else img

In [ ]:
#@title generation
numberToGenerate = 16

final, history = sample_ddim(
    model, sampling_scheduler,
    n=numberToGenerate, image_size=IMAGE_SIZE, keep_history=True
)
final = (final.clamp(-1, 1) + 1) / 2              # [-1,1] -> [0,1]
final = final.cpu()

figure = plt.figure(figsize=(12, 12))
cols, rows = 4, 4
for i in range(numberToGenerate):
    sample_img = final[i]
    figure.add_subplot(rows, cols, i + 1)
    plt.imshow(sample_img.permute(1, 2, 0).numpy())   # CHW -> HWC
    plt.title(f"Generated #{i+1}")
    plt.axis('off')

plt.suptitle("Final Generated Anime Faces (DDIM)", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
#@title step-tracking state (run once)
used_step_counts = set()          # remembers which step counts you've already tried
STEP_MIN, STEP_MAX = 20, 200      # range to draw from

34 138 90 61 70 194 200

In [ ]:
#@title generation
import random

# pick a step count we haven't used yet
available = [s for s in range(STEP_MIN, STEP_MAX + 1) if s not in used_step_counts]
if not available:
    print("All step counts in range used — clearing the list and starting over.")
    used_step_counts.clear()
    available = list(range(STEP_MIN, STEP_MAX + 1))

num_steps = random.choice(available)
used_step_counts.add(num_steps)
print(f"Sampling with {num_steps} DDIM steps   (used so far: {sorted(used_step_counts)})")

sampling_scheduler.set_timesteps(num_steps)
numberToGenerate = 16

final, history = sample_ddim(
    model, sampling_scheduler,
    n=numberToGenerate, image_size=IMAGE_SIZE, keep_history=True
)
final = (final.clamp(-1, 1) + 1) / 2              # [-1,1] -> [0,1]
final = final.cpu()

figure = plt.figure(figsize=(12, 12))
cols, rows = 4, 4
for i in range(numberToGenerate):
    sample_img = final[i]
    figure.add_subplot(rows, cols, i + 1)
    plt.imshow(sample_img.permute(1, 2, 0).numpy())   # CHW -> HWC
    plt.title(f"Generated #{i+1}")
    plt.axis('off')

plt.suptitle(f"Final Generated Anime Faces (DDIM, {num_steps} steps)", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# ---- Animate the denoising of one image ----
random_index = 10                                 # which of the 16 to animate
num_frames = len(history)                         # ~51 frames for 50 DDIM steps

fig = plt.figure(figsize=(8, 8))
plt.axis("off")
ims = []
for i in range(num_frames):
    frame = history[i][random_index]              # (3, H, W) tensor
    frame = torch.clamp((frame + 1) / 2, 0, 1)    # [-1,1] -> [0,1]
    im = plt.imshow(frame.permute(1, 2, 0).numpy(), animated=True)
    ims.append([im])

animate = animation.ArtistAnimation(fig, ims, interval=80, blit=True, repeat_delay=1000)
rc('animation', html='jshtml')
animate

In [ ]:
from getpass import getpass

# Prompts you to paste the token — it won't be echoed or saved in notebook output
HF_TOKEN = getpass("Enter your Hugging Face token: ")

In [ ]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [ ]:
import os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["DISABLE_TELEMETRY"] = "1"

In [ ]:
from huggingface_hub import HfApi, create_repo

repo_id = "AnasKAN/attn_ddim_anime2"          # new repo, DDIM-named

# 1. Save the model locally first (fast, no network)
model.save_pretrained("attn_ddim_anime_local2")

# 2. Create the repo if it doesn't exist (public by default)
create_repo(repo_id, exist_ok=True)

# 3. Upload — overrides same-named files, returns cleanly in Colab
api = HfApi()
api.upload_folder(
    folder_path="attn_ddim_anime_local2",
    repo_id=repo_id,
    commit_message="Upload attn DDIM anime UNet",
)
print("Done:", f"https://huggingface.co/{repo_id}")

## Contributed by:

#### [Anas](https://github.com/AnasKAN)

<img src="https://drive.google.com/uc?id=1sRRSwuPqoVpdVmM5xZ88kbShTI_RvE_1" width="100" height="100"/>